# Portfolio Project: Automobile Price Analysis and Regression Modeling 

---

# 01 — Data Importing & Initial Understanding
## Automobile Pricing Analytics

### Purpose
This notebook establishes the analytical foundation for the project by loading the raw automobile dataset, documenting its provenance and structure, and performing an initial quality assessment.

The broader project investigates how automobile characteristics are associated with vehicle price and how well alternative regression specifications explain price variation.

### Objectives
1. Load the raw dataset reproducibly.
2. Assign the documented variable names to the headerless source file.
3. Verify the dataset dimensions and structure.
4. Inspect data types, sample observations, and category coverage.
5. Audit missing-value markers and duplicate records.
6. Identify issues that should be addressed in the next notebook.

## 1. Dataset provenance

The data used in this project are the **Automobile** dataset from the **UCI Machine Learning Repository**. UCI describes the dataset as originating from the **1985 Ward's Automotive Yearbook** and containing automobile specifications together with insurance-risk and normalized-loss information. The repository reports **205 observations**, **25 input features**, and a regression task; the raw file used here also includes the `price` field, giving 26 columns in total.

The local `auto.csv` file corresponds to the copy distributed through the UCI Machine Learning Repository.

**Primary source**  
Schlimmer, J. (1985). *Automobile* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5B01C  
UCI metadata: https://archive.ics.uci.edu/dataset/10/automobile

> This notebook treats the downloaded CSV as raw source data. Missing-value treatment, type correction, transformation, normalization, and feature engineering are intentionally deferred to Notebook 02.

## 2. Import libraries and locate the raw data

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
# Search a few common locations so the notebook remains portable when the
# repository is run either from its root directory or from a notebooks/ folder.
candidate_paths = [
    Path("../data/auto.csv"),
    Path("data/auto.csv"),
    Path("auto.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "auto.csv was not found. Place it in ../data/, data/, or the current directory."
    )

print(f"Dataset located at: {data_path.resolve()}")

Dataset located at: /mnt/data/auto.csv


## 3. Define the documented schema and load the dataset

The source CSV does not contain a header row, so the variable names are supplied explicitly using the schema used in the UCI documentation.

In [3]:
columns = [
    "symboling", "normalized-losses", "make", "fuel-type", "aspiration",
    "num-of-doors", "body-style", "drive-wheels", "engine-location",
    "wheel-base", "length", "width", "height", "curb-weight",
    "engine-type", "num-of-cylinders", "engine-size", "fuel-system",
    "bore", "stroke", "compression-ratio", "horsepower", "peak-rpm",
    "city-mpg", "highway-mpg", "price"
]

raw_df = pd.read_csv(data_path, header=None, names=columns)

print(f"Rows: {raw_df.shape[0]:,}")
print(f"Columns: {raw_df.shape[1]}")

Rows: 205
Columns: 26


### Structural verification
The expected raw structure is 205 observations and 26 columns (25 descriptive/input variables plus `price`). The checks below fail loudly if the local file does not match that expected structure.

In [4]:
expected_shape = (205, 26)
assert raw_df.shape == expected_shape, (
    f"Unexpected dataset shape: {raw_df.shape}; expected {expected_shape}."
)
assert list(raw_df.columns) == columns

print("Schema and dimensions verified successfully.")

Schema and dimensions verified successfully.


## 4. Preview the observations

In [5]:
raw_df.head(10)

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,length,width,height,curb-weight,engine-type,num-of-cylinders,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.60,168.80,64.10,48.80,2548,dohc,four,130,mpfi,3.47,2.68,9.00,111,5000,21,27,13495
1,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.60,168.80,64.10,48.80,2548,dohc,four,130,mpfi,3.47,2.68,9.00,111,5000,21,27,16500
2,1,?,alfa-romero,gas,std,two,hatchback,rwd,front,94.50,171.20,65.50,52.40,2823,ohcv,six,152,mpfi,2.68,3.47,9.00,154,5000,19,26,16500
3,2,164,audi,gas,std,four,sedan,fwd,front,99.80,176.60,66.20,54.30,2337,ohc,four,109,mpfi,3.19,3.40,10.00,102,5500,24,30,13950
4,2,164,audi,gas,std,four,sedan,4wd,front,99.40,176.60,66.40,54.30,2824,ohc,five,136,mpfi,3.19,3.40,8.00,115,5500,18,22,17450
5,2,?,audi,gas,std,two,sedan,fwd,front,99.80,177.30,66.30,53.10,2507,ohc,five,136,mpfi,3.19,3.40,8.50,110,5500,19,25,15250
6,1,158,audi,gas,std,four,sedan,fwd,front,105.80,192.70,71.40,55.70,2844,ohc,five,136,mpfi,3.19,3.40,8.50,110,5500,19,25,17710
7,1,?,audi,gas,std,four,wagon,fwd,front,105.80,192.70,71.40,55.70,2954,ohc,five,136,mpfi,3.19,3.40,8.50,110,5500,19,25,18920
8,1,158,audi,gas,turbo,four,sedan,fwd,front,105.80,192.70,71.40,55.90,3086,ohc,five,131,mpfi,3.13,3.40,8.30,140,5500,17,20,23875
9,0,?,audi,gas,turbo,two,hatchback,4wd,front,99.50,178.20,67.90,52.00,3053,ohc,five,131,mpfi,3.13,3.40,7.00,160,5500,16,22,?


The first observations confirm that the file combines numerical automobile specifications with categorical design and configuration attributes. The raw missing-value marker `?` is also visible in variables such as `normalized-losses`.

## 5. Initial data structure

In [6]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   symboling          205 non-null    int64  
 1   normalized-losses  205 non-null    object 
 2   make               205 non-null    object 
 3   fuel-type          205 non-null    object 
 4   aspiration         205 non-null    object 
 5   num-of-doors       205 non-null    object 
 6   body-style         205 non-null    object 
 7   drive-wheels       205 non-null    object 
 8   engine-location    205 non-null    object 
 9   wheel-base         205 non-null    float64
 10  length             205 non-null    float64
 11  width              205 non-null    float64
 12  height             205 non-null    float64
 13  curb-weight        205 non-null    int64  
 14  engine-type        205 non-null    object 
 15  num-of-cylinders   205 non-null    object 
 16  engine-size        205 non

Several columns that are conceptually numeric may appear as `object` because the raw file uses `?` to represent missing values. This is a data-quality issue to resolve in Notebook 02.

In [7]:
dtype_summary = (
    raw_df.dtypes.astype(str)
    .value_counts()
    .rename_axis("pandas_dtype")
    .to_frame("column_count")
)
dtype_summary

,column_count
pandas_dtype,
object,16
int64,5
float64,5


## 6. Raw missing-value marker audit

In [8]:
missing_marker_count = raw_df.eq("?").sum().sort_values(ascending=False)
missing_marker_count = missing_marker_count[missing_marker_count > 0]

missing_audit = pd.DataFrame({
    "missing_marker_count": missing_marker_count,
    "percent_of_rows": (missing_marker_count / len(raw_df) * 100).round(2)
})
missing_audit

,missing_marker_count,percent_of_rows
normalized-losses,41,20.00
stroke,4,1.95
price,4,1.95
bore,4,1.95
horsepower,2,0.98
peak-rpm,2,0.98
num-of-doors,2,0.98


In [9]:
print(f"Columns containing '?' markers: {len(missing_audit)}")
print(f"Total '?' markers in the raw dataset: {int(missing_marker_count.sum())}")

Columns containing '?' markers: 7
Total '?' markers in the raw dataset: 59


### Initial implication
The raw dataset contains explicit missing-value markers in multiple variables. Because the number and context of missing values differ by feature, Notebook 02 will evaluate treatment decisions variable by variable rather than applying one blanket rule.

## 7. Duplicate-record audit

In [10]:
duplicate_rows = raw_df.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_rows}")

Exact duplicate rows: 0


Exact duplicates are reported here as part of data-quality assessment only. Any decision to remove records belongs to the cleaning workflow in Notebook 02.

## 8. Descriptive overview of variables already stored as numeric

In [11]:
raw_df.describe().T

,count,mean,std,min,25%,50%,75%,max
symboling,205.00,0.83,1.25,-2.00,0.00,1.00,2.00,3.00
wheel-base,205.00,98.76,6.02,86.60,94.50,97.00,102.40,120.90
length,205.00,174.05,12.34,141.10,166.30,173.20,183.10,208.10
width,205.00,65.91,2.15,60.30,64.10,65.50,66.90,72.30
height,205.00,53.72,2.44,47.80,52.00,54.10,55.50,59.80
curb-weight,205.00,"2,555.57",520.68,"1,488.00","2,145.00","2,414.00","2,935.00","4,066.00"
engine-size,205.00,126.91,41.64,61.00,97.00,120.00,141.00,326.00
compression-ratio,205.00,10.14,3.97,7.00,8.60,9.00,9.40,23.00
city-mpg,205.00,25.22,6.54,13.00,19.00,24.00,30.00,49.00
highway-mpg,205.00,30.75,6.89,16.00,25.00,30.00,34.00,54.00


`describe()` currently summarizes only columns that Pandas has inferred as numeric. Variables containing `?` may be excluded even when their underlying meaning is numerical. A complete numerical summary should therefore be produced only after missing markers and data types are handled consistently in Notebook 02.

## 9. Categorical coverage

In [12]:
categorical_columns = raw_df.select_dtypes(include="object").columns

category_overview = pd.DataFrame({
    "unique_values_including_missing_marker": raw_df[categorical_columns].nunique(dropna=False),
    "most_frequent_value": raw_df[categorical_columns].mode(dropna=False).iloc[0],
    "frequency_of_most_common": [
        raw_df[col].value_counts(dropna=False).iloc[0] for col in categorical_columns
    ],
}).sort_values("unique_values_including_missing_marker", ascending=False)

category_overview

,unique_values_including_missing_marker,most_frequent_value,frequency_of_most_common
price,187,?,4
horsepower,60,68,19
normalized-losses,52,?,41
bore,39,3.62,23
stroke,37,3.40,20
peak-rpm,24,5500,37
make,22,toyota,32
fuel-system,8,mpfi,94
engine-type,7,ohc,148
num-of-cylinders,7,four,159


Because some conceptually numeric columns are temporarily stored as `object`, this table is an **import-stage diagnostic**, not a final categorical-variable definition. Notebook 02 will correct the schema before substantive analysis.

## 10. Target-variable (`price`) availability

In [13]:
raw_price_missing = raw_df["price"].eq("?").sum()
raw_price_available = len(raw_df) - raw_price_missing

print(f"Price values available: {raw_price_available}")
print(f"Price values marked missing ('?'): {raw_price_missing}")

Price values available: 201
Price values marked missing ('?'): 4


In [14]:
# Temporary conversion for an import-stage descriptive check only.
# raw_df itself is left unchanged.
price_numeric_preview = pd.to_numeric(raw_df["price"].replace("?", np.nan), errors="coerce")
price_numeric_preview.describe().to_frame("price")

,price
count,201.00
mean,"13,207.13"
std,"7,947.07"
min,"5,118.00"
25%,"7,775.00"
50%,"10,295.00"
75%,"16,500.00"
max,"45,400.00"


Price is the response variable for the later regression analysis. Observations without a recorded price cannot contribute directly to supervised price modeling, but the formal decision about how to handle them is deliberately documented in Notebook 02.

## 11. Initial assessment and handoff to cleaning

The importing stage establishes the following:

- The local source file matches the expected **205 × 26** raw automobile dataset structure.
- The source file is headerless, so the documented schema must be assigned explicitly.
- The data contain a mixture of numerical specifications and categorical automobile characteristics.
- The raw file uses `?` as an explicit missing-value marker in several fields.
- Some conceptually numerical variables are therefore imported as `object` and require type correction.
- The `price` target also contains missing observations that require an explicit modeling decision.
- Exact duplicate records have been audited but not modified at this stage.

### Next step — Notebook 02: Data Cleaning & Feature Preparation
Notebook 02 will convert the missing-value markers into a consistent representation, correct data types, justify missing-data treatments, and prepare analysis-ready variables. Keeping these operations separate from importing preserves the raw data and makes the analytical workflow easier to audit and reproduce.